# Derm-Vision: ISIC 2019 Skin Lesion Classification — Training

This notebook runs the full training pipeline on Google Colab with GPU.

Data is downloaded directly from Kaggle to Colab's local disk — no Google Drive storage needed for the dataset.

## 1. GPU Check

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    !nvidia-smi --query-gpu=memory.total --format=csv,noheader

## 2. Clone Repo & Install Dependencies

In [ ]:
!git clone https://github.com/Lambert-Nguyen/derm-vision.git /content/derm-vision
%cd /content/derm-vision

In [ ]:
!pip install -q timm albumentations wandb grad-cam kaggle

## 3. Download ISIC 2019 from Kaggle

Downloads directly to Colab's local disk (~100GB free) — no Drive space needed.

**Setup**: Go to https://www.kaggle.com/settings → copy your **Username** and **API Key**, then paste them below.

In [ ]:
import os
import json
from getpass import getpass

# Prompt for credentials (won't be displayed or saved in the notebook)
kaggle_username = input("Kaggle username: ")
kaggle_key = getpass("Kaggle API key: ")

# Write kaggle.json
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_json_path, "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(kaggle_json_path, 0o600)
print("Kaggle API configured.")

In [ ]:
# Download ISIC 2019 dataset
!kaggle datasets download -d andrewmvd/isic-2019 -p /content/isic-data --unzip

# List downloaded files
!ls /content/isic-data/

In [ ]:
# Symlink data into the project directory
import os
import glob

os.makedirs("data/raw", exist_ok=True)

# Auto-detect the data layout from Kaggle
kaggle_dir = "/content/isic-data"

# Find the images directory
img_dir = None
for candidate in [
    os.path.join(kaggle_dir, "ISIC_2019_Training_Input"),
    os.path.join(kaggle_dir, "ISIC_2019_Training_Input", "ISIC_2019_Training_Input"),
]:
    if os.path.isdir(candidate) and glob.glob(os.path.join(candidate, "*.jpg")):
        img_dir = candidate
        break

if img_dir is None:
    raise FileNotFoundError(
        f"Could not find images in {kaggle_dir}. Contents: {os.listdir(kaggle_dir)}"
    )

# Find CSV files
gt_matches = glob.glob(os.path.join(kaggle_dir, "**", "*GroundTruth*"), recursive=True)
meta_matches = glob.glob(os.path.join(kaggle_dir, "**", "*Metadata*"), recursive=True)
if not gt_matches or not meta_matches:
    raise FileNotFoundError(f"Missing CSVs in {kaggle_dir}. Contents: {os.listdir(kaggle_dir)}")
gt_csv = gt_matches[0]
meta_csv = meta_matches[0]

# Create symlinks
links = [
    (img_dir, "data/raw/ISIC_2019_Training_Input/ISIC_2019_Training_Input"),
    (gt_csv, "data/raw/ISIC_2019_Training_GroundTruth.csv"),
    (meta_csv, "data/raw/ISIC_2019_Training_Metadata.csv"),
]

for src, dst in links:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst) or os.path.islink(dst):
        os.remove(dst)
    os.symlink(src, dst)
    print(f"{dst} -> {src}")

# Verify
n_images = len(glob.glob("data/raw/ISIC_2019_Training_Input/ISIC_2019_Training_Input/*.jpg"))
print(f"\nFound {n_images} images.")

## 4. Create Data Splits

In [ ]:
if not os.path.exists("data/splits/train.csv"):
    !python scripts/create_splits.py
else:
    print("Splits already exist, skipping.")

## 5. Mount Drive for Checkpoints Only

Checkpoints (~50MB) are saved to Drive so they persist across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import yaml

# Read config, update output paths only, rewrite preserving structure
config_path = "configs/config.yaml"
with open(config_path) as f:
    config_text = f.read()
    cfg = yaml.safe_load(config_text)

# Save checkpoints & results to Drive (small files only)
drive_output = "/content/drive/MyDrive/derm-vision-outputs"
os.makedirs(os.path.join(drive_output, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(drive_output, "results"), exist_ok=True)

# Replace only the output paths via string replacement to preserve comments
config_text = config_text.replace(
    f"checkpoint_dir: {cfg['output']['checkpoint_dir']}",
    f"checkpoint_dir: {os.path.join(drive_output, 'checkpoints')}"
)
config_text = config_text.replace(
    f"results_dir: {cfg['output']['results_dir']}",
    f"results_dir: {os.path.join(drive_output, 'results')}"
)

with open(config_path, "w") as f:
    f.write(config_text)

# Reload to verify
with open(config_path) as f:
    cfg = yaml.safe_load(f)
print("Checkpoints will save to Drive (minimal storage needed).")
print(f"  checkpoint_dir: {cfg['output']['checkpoint_dir']}")
print(f"  results_dir: {cfg['output']['results_dir']}")

## 6. Login to W&B

In [ ]:
import wandb
wandb.login()

## 7. Train

In [ ]:
!python -m src.train --config configs/config.yaml

## 8. Evaluate on Test Set

In [ ]:
from src.dataset import ISICDataset, CLASS_NAMES
from src.transforms import get_val_transforms
from src.models.efficientnet import EfficientNetB3Classifier
from src.evaluate import compute_metrics, print_classification_report, plot_confusion_matrix
from torch.utils.data import DataLoader

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load best model
model = EfficientNetB3Classifier(
    num_classes=cfg["data"]["num_classes"],
    pretrained=False,
    dropout=cfg["model"]["dropout"],
).to(device)
ckpt_path = os.path.join(cfg["output"]["checkpoint_dir"], "best_model.pth")
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

# Test dataset
test_dataset = ISICDataset(
    image_dir=cfg["data"]["data_dir"],
    labels_csv=os.path.join(cfg["data"]["splits_dir"], "test.csv"),
    transform=get_val_transforms(cfg["data"]["image_size"]),
)
test_loader = DataLoader(test_dataset, batch_size=cfg["training"]["batch_size"], num_workers=2)

# Run inference
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        all_preds.extend(outputs.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.tolist())

# Metrics
metrics = compute_metrics(all_labels, all_preds)
print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
print(f"Weighted F1:       {metrics['weighted_f1']:.4f}")
print()
print_classification_report(all_labels, all_preds)

# Confusion matrix
save_path = os.path.join(cfg["output"]["results_dir"], "confusion_matrix.png")
plot_confusion_matrix(all_labels, all_preds, save_path=save_path)